# Stimulus Degradation: Probing the Trained Network

Weights are **frozen** (no retraining). We test how the network behaves under:

| Manipulation | Training value | Test values |
|---|---|---|
| Cue strength | 1.0 (coords ±0.40) | 0.8, 0.5, 0.3, 0.1 |
| Target strength | 1.0 | 0.8, 0.5, 0.3, 0.1 |
| Target duration | 100 ms | 80, 60, 40, 20 ms |
| Recurrent noise | σ=0.05 | σ=0.10, 0.20, 0.40 |

Outcome metrics: p_correct, p_miss, p_false_alarm, p_abort, mean RT

In [ ]:
import sys, os, pathlib
ROOT = next(p for p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]
            if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import numpy as np
import torch
import matplotlib.pyplot as plt
from collections import defaultdict
import copy

from src.model import BioLeakyRNN
from src.env import CuedTargetWithDistractorsV3
from src.analysis import collect_trials

device = "cpu"
N_TRIALS = 1000  # per condition
print(f"Trials per condition: {N_TRIALS}")

## Load trained model (weights frozen)

In [ ]:
def make_model(sigma_rec=0.05):
    return BioLeakyRNN(
        input_size=7,
        hidden_size=128,
        output_size=2,
        dt=20.0,
        tau=100.0,
        activation="softplus",
        sigma_rec=sigma_rec,
        use_ei=True,
        exc_ratio=0.7,
        use_dale=True,
        mask_seed=42,
    )


ckpt = torch.load("checkpoints/stage2.pt", weights_only=True)["state_dict"]

base_model = make_model(sigma_rec=0.05)
base_model.load_state_dict(ckpt)
base_model.eval()
print("Loaded stage2 checkpoint.")

## Helper: run a condition and extract outcome stats

In [ ]:
def run_condition(model, env_fn, n_trials=N_TRIALS):
    """Returns dict of outcome rates and mean RT."""
    trials = collect_trials(model, env_fn, n_trials=n_trials, device=device)
    counts = defaultdict(int)
    rts = []
    for tr in trials:
        counts[tr["train_outcome"]] += 1
        if tr["train_outcome"] == "correct" and tr["rt_ms"] is not None:
            rts.append(tr["rt_ms"])
    total = len(trials)
    return {
        "p_correct": counts["correct"] / total,
        "p_miss": counts["miss"] / total,
        "p_false_alarm": counts["false_alarm"] / total,
        "p_abort": counts["abort"] / total,
        "rt_mean": np.mean(rts) if rts else np.nan,
        "rt_std": np.std(rts) if rts else np.nan,
        "n": total,
    }


def print_stats(label, stats):
    print(
        f"{label:40s}  "
        f"correct={stats['p_correct']:.3f}  "
        f"miss={stats['p_miss']:.3f}  "
        f"FA={stats['p_false_alarm']:.3f}  "
        f"abort={stats['p_abort']:.3f}  "
        f"RT={stats['rt_mean']:.0f}ms"
    )


def make_env_base():
    return CuedTargetWithDistractorsV3(
        dt=20,
        cue_strength=1.0,
        target_strength=1.0,
        p_distractor_trial=0.6,
        distractor_strength=1.0,
    )


print("Running baseline...")
baseline = run_condition(base_model, make_env_base)
print_stats("BASELINE (cue=1.0, tgt=1.0, 100ms, σ=0.05)", baseline)

## Experiment 1: Cue strength sweep

Training: `cue_strength=1.0` → coordinates ±0.40  
Test: reduce to 0.8, 0.5, 0.3, 0.1  
**Prediction:** weaker cue → network cannot reliably identify target location → more misses (responds at wrong time) and more FAs (confuses distractor with target)

In [ ]:
cue_strengths = [1.0, 0.8, 0.5, 0.3, 0.1]
cue_results = {}

for cs in cue_strengths:

    def env_fn(cs=cs):
        return CuedTargetWithDistractorsV3(
            dt=20,
            cue_strength=cs,
            target_strength=1.0,
            p_distractor_trial=0.6,
            distractor_strength=1.0,
        )

    cue_results[cs] = run_condition(base_model, env_fn)
    print_stats(f"cue_strength={cs:.1f}", cue_results[cs])

## Experiment 2: Target strength sweep

Training: `target_strength=1.0` (coordinates ±1.00)  
Test: 0.8, 0.5, 0.3, 0.1  
**Prediction:** weaker target → harder to detect → more misses, RT increases

In [ ]:
target_strengths = [1.0, 0.8, 0.5, 0.3, 0.1]
tgt_strength_results = {}

for ts in target_strengths:

    def env_fn(ts=ts):
        return CuedTargetWithDistractorsV3(
            dt=20,
            cue_strength=1.0,
            target_strength=ts,
            p_distractor_trial=0.6,
            distractor_strength=1.0,
        )

    tgt_strength_results[ts] = run_condition(base_model, env_fn)
    print_stats(f"target_strength={ts:.1f}", tgt_strength_results[ts])

## Experiment 3: Target duration sweep

Training: 100 ms target  
Test: 80, 60, 40, 20 ms  
**Prediction:** shorter target = fewer timesteps of sensory evidence → more misses

In [ ]:
target_durations = [100, 80, 60, 40, 20]  # ms
tgt_dur_results = {}

for dur in target_durations:

    def env_fn(dur=dur):
        env = CuedTargetWithDistractorsV3(
            dt=20,
            cue_strength=1.0,
            target_strength=1.0,
            p_distractor_trial=0.6,
            distractor_strength=1.0,
        )
        env.timing["target"] = dur
        return env

    tgt_dur_results[dur] = run_condition(base_model, env_fn)
    print_stats(f"target_duration={dur}ms", tgt_dur_results[dur])

## Experiment 4: Recurrent noise sweep

Same weights, but `sigma_rec` increased at test time.  
**Prediction:** more noise → state space trajectories become less precise → more misses and FAs  
Also makes the network **more biological** (rot_index ↓, Spearman ρ ↓)

In [ ]:
sigma_values = [0.05, 0.10, 0.20, 0.40, 0.80]
noise_results = {}

for sigma in sigma_values:
    noisy_model = make_model(sigma_rec=sigma)
    noisy_model.load_state_dict(ckpt)  # same weights
    # Override sigma_eff buffer to match new sigma
    alpha = float(noisy_model.alpha)
    import math

    new_sigma_eff = math.sqrt(2.0 / alpha) * sigma
    noisy_model.sigma_eff.fill_(new_sigma_eff)
    noisy_model.sigma_rec = sigma
    noisy_model.eval()
    # Force training mode so noise is active
    noisy_model.train()

    noise_results[sigma] = run_condition(noisy_model, make_env_base)
    print_stats(f"sigma_rec={sigma:.2f}", noise_results[sigma])

## Experiment 5: Combined — weak cue + noise (most biological condition)

In [ ]:
combined_conditions = [
    {"cue": 1.0, "tgt": 1.0, "dur": 100, "sigma": 0.05, "label": "baseline"},
    {"cue": 0.3, "tgt": 1.0, "dur": 100, "sigma": 0.05, "label": "weak cue"},
    {"cue": 1.0, "tgt": 0.3, "dur": 100, "sigma": 0.05, "label": "weak target"},
    {"cue": 1.0, "tgt": 1.0, "dur": 40, "sigma": 0.05, "label": "short target (40ms)"},
    {
        "cue": 0.3,
        "tgt": 0.3,
        "dur": 100,
        "sigma": 0.05,
        "label": "weak cue + weak target",
    },
    {"cue": 0.3, "tgt": 1.0, "dur": 100, "sigma": 0.20, "label": "weak cue + noise"},
    {"cue": 0.3, "tgt": 0.3, "dur": 40, "sigma": 0.20, "label": "all degraded"},
]

combined_results = []
for c in combined_conditions:
    noisy_model = make_model(sigma_rec=c["sigma"])
    noisy_model.load_state_dict(ckpt)
    import math

    noisy_model.sigma_eff.fill_(math.sqrt(2.0 / float(noisy_model.alpha)) * c["sigma"])
    noisy_model.sigma_rec = c["sigma"]
    noisy_model.train()  # activate noise

    def env_fn(c=c):
        env = CuedTargetWithDistractorsV3(
            dt=20,
            cue_strength=c["cue"],
            target_strength=c["tgt"],
            p_distractor_trial=0.6,
            distractor_strength=1.0,
        )
        env.timing["target"] = c["dur"]
        return env

    res = run_condition(noisy_model, env_fn)
    combined_results.append({**c, **res})
    print_stats(c["label"], res)

## Summary plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = ["p_correct", "p_miss", "p_false_alarm", "p_abort"]
titles = ["p(correct)", "p(miss)", "p(false alarm)", "p(abort)"]
colors = ["steelblue", "crimson", "orange", "gray"]

for ax, metric, title, col in zip(axes.flat, metrics, titles, colors):
    # Cue strength
    x = cue_strengths
    y = [cue_results[cs][metric] for cs in cue_strengths]
    ax.plot(x, y, "o-", color=col, lw=2, label="cue strength")

    # Target strength (dashed)
    y2 = [tgt_strength_results[ts][metric] for ts in target_strengths]
    ax.plot(
        target_strengths, y2, "s--", color=col, lw=1.5, alpha=0.7, label="target strength"
    )

    ax.axhline(baseline[metric], color="black", ls=":", lw=1, alpha=0.5, label="baseline")
    ax.set_xlabel("Strength (relative to training)")
    ax.set_ylabel(title)
    ax.set_title(title)
    ax.set_xlim([-0.05, 1.1])
    ax.set_ylim([-0.02, 1.02])
    if metric == "p_correct":
        ax.legend(fontsize=8)

plt.suptitle("Outcome rates vs stimulus degradation\n(solid=cue, dashed=target)", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Left: noise sweep ---
ax = axes[0]
x = sigma_values
for metric, col, label in [
    ("p_correct", "steelblue", "correct"),
    ("p_miss", "crimson", "miss"),
    ("p_false_alarm", "orange", "FA"),
    ("p_abort", "gray", "abort"),
]:
    y = [noise_results[s][metric] for s in sigma_values]
    ax.plot(x, y, "o-", lw=2, color=col, label=label)
ax.axvline(0.05, color="black", ls=":", lw=1, alpha=0.4, label="training σ")
ax.set_xlabel("sigma_rec (noise level)")
ax.set_ylabel("proportion of trials")
ax.set_title("Outcome rates vs recurrent noise")
ax.legend(fontsize=8)

# --- Right: target duration ---
ax2 = axes[1]
for metric, col, label in [
    ("p_correct", "steelblue", "correct"),
    ("p_miss", "crimson", "miss"),
    ("p_false_alarm", "orange", "FA"),
    ("p_abort", "gray", "abort"),
]:
    y = [tgt_dur_results[d][metric] for d in target_durations]
    ax2.plot(target_durations, y, "o-", lw=2, color=col, label=label)
ax2.axvline(100, color="black", ls=":", lw=1, alpha=0.4, label="training dur")
ax2.set_xlabel("target duration (ms)")
ax2.set_ylabel("proportion of trials")
ax2.set_title("Outcome rates vs target duration")
ax2.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Combined conditions bar plot
labels = [c["label"] for c in combined_conditions]
x = np.arange(len(labels))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 5))
for i, (metric, col, name) in enumerate(
    [
        ("p_correct", "steelblue", "correct"),
        ("p_miss", "crimson", "miss"),
        ("p_false_alarm", "orange", "FA"),
        ("p_abort", "gray", "abort"),
    ]
):
    y = [r[metric] for r in combined_results]
    ax.bar(x + i * width, y, width, label=name, color=col, alpha=0.85)

ax.set_xticks(x + 1.5 * width)
ax.set_xticklabels(labels, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("proportion of trials")
ax.set_title("Combined degradation conditions")
ax.set_ylim([0, 1.05])
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# RT as a function of cue & target strength
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, results, x_vals, xlabel in [
    (axes[0], cue_results, cue_strengths, "cue strength"),
    (axes[1], tgt_strength_results, target_strengths, "target strength"),
    (axes[2], noise_results, sigma_values, "sigma_rec"),
]:
    rt_means = [results[v]["rt_mean"] for v in x_vals]
    rt_stds = [results[v]["rt_std"] for v in x_vals]
    ax.errorbar(
        x_vals, rt_means, yerr=rt_stds, fmt="o-", color="steelblue", lw=2, capsize=4
    )
    ax.set_xlabel(xlabel)
    ax.set_ylabel("mean RT (ms)")
    ax.set_title(f"RT vs {xlabel}")

plt.tight_layout()
plt.show()

## Summary table

In [ ]:
print(
    f"{'Condition':<42} {'correct':>8} {'miss':>8} {'FA':>8} {'abort':>8} {'RT(ms)':>8}"
)
print("-" * 82)
print_stats("BASELINE", baseline)
print()
print("-- Cue strength --")
for cs in cue_strengths:
    print_stats(f"  cue={cs:.1f}", cue_results[cs])
print()
print("-- Target strength --")
for ts in target_strengths:
    print_stats(f"  target_strength={ts:.1f}", tgt_strength_results[ts])
print()
print("-- Target duration --")
for dur in target_durations:
    print_stats(f"  target_dur={dur}ms", tgt_dur_results[dur])
print()
print("-- Noise --")
for sigma in sigma_values:
    print_stats(f"  sigma={sigma:.2f}", noise_results[sigma])
print()
print("-- Combined --")
for c, r in zip(combined_conditions, combined_results):
    print_stats(f"  {c['label']}", r)